# Прогнозирование цен на жилье в Турции

Цели проекта: Проанализировать рынок недвижимости Турции на основе датасета Zingat и подготовить данные для прогнозирования цен.

In [1]:
import pandas as pd

In [76]:
df = pd.read_csv('real_estate_data.csv',low_memory=False)

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 403487 entries, 0 to 403486
Data columns (total 17 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 403487 non-null  int64  
 1   type               403487 non-null  object 
 2   sub_type           403487 non-null  object 
 3   start_date         403487 non-null  object 
 4   end_date           266298 non-null  object 
 5   listing_type       403487 non-null  int64  
 6   tom                403487 non-null  int64  
 7   building_age       376097 non-null  object 
 8   total_floor_count  375466 non-null  object 
 9   floor_no           368191 non-null  object 
 10  room_count         403487 non-null  object 
 11  size               257481 non-null  float64
 12  address            403487 non-null  object 
 13  furnished          0 non-null       float64
 14  heating_type       375517 non-null  object 
 15  price              402772 non-null  float64
 16  pr

Описание:
- Zingat — турецкая платформа по недвижимости
- Данные содержат объявления о продаже и аренде
- Датасет включает 403,487 объявлений
- Каждое объявление содержит информацию о:
  • Типе недвижимости (жильё, офис, коммерческое)
  • Местоположении (город, район, улица)
  • Характеристиках (площадь, комнаты, этаж, возраст)
  • Цене и валюте
  • Времени на рынке

1. КАТЕГОРИАЛЬНЫЕ (объектные) признаки:
   • тип           → тип недвижимости (Konut, Villa, Ofis...)
   • подтип        → подтип объекта (Daire, Müstakil Ev...)
   • тип_объявления → продажа или аренда (Satılık, Kiralık)
   • адрес         → полный адрес (город/район/улица)
   • город         → город
   • район         → район
   • улица         → улица
   • отопление     → тип отопления
   • валюта        → валюта цены

2. ЧИСЛОВЫЕ (количественные) признаки:
   • площадь       → площадь в м²
   • комнат_всего  → общее количество комнат
   • спальни       → количество спален
   • гостиные      → количество гостиных
   • всего_этажей  → количество этажей в здании
   • этаж          → этаж квартиры
   • возраст_здания → возраст здания
   • время_на_рынке → количество дней объявления на сайте

3. ЦЕЛЕВАЯ ПЕРЕМЕННАЯ:
   • цена_usd      → цена объекта в долларах США

In [79]:
df.describe()

,id,listing_type,tom,size,furnished,price
count,403487.00000,403487.000000,403487.000000,257481.000000,0.0,4.027720e+05
mean,201744.00000,1.294235,57.022739,279.349094,NaN,3.546417e+05
std,116476.80837,0.467733,44.358933,9429.195331,NaN,4.809503e+06
min,1.00000,1.000000,0.000000,1.000000,NaN,-2.500000e+02
25%,100872.50000,1.000000,29.000000,85.000000,NaN,2.500000e+03
50%,201744.00000,1.000000,40.000000,110.000000,NaN,1.990000e+05
75%,302615.50000,2.000000,90.000000,140.000000,NaN,3.420000e+05
max,403487.00000,3.000000,180.000000,948235.000000,NaN,2.000000e+09


- Тип объявления представлен тремя значениями: 1 (продажа), 2 (аренда) и 3 (другое). Среднее значение составляет 1.29, что указывает на преобладание объявлений о продаже. Поскольку это категориальный признак, числовые статистики (среднее, медиана) 
не имеют содержательного смысла и требуют дополнительного анализа распределения.

- Время на рынке варьируется от 0 до 180 дней. Среднее значение составляет 57 дней, медиана — 40 дней. 
Это говорит о том, что большинство объектов находятся на рынке около 1.5–2 месяцев.

- Площадь объектов варьируется от 1 до 948 235 м², что указывает на наличие аномалий (выбросов). Среднее значение (279 м²) значительно выше медианы (110 м²), что подтверждает влияние экстремально больших значений

- Столбец furnished полностью пустой (все значения NaN). Он не содержит информации и подлежит удалению

- Цена варьируется от -250 до 2 000 000 000, что явно указывает на наличие ошибок и аномалий. Отрицательные значения и экстремально высокие цены требуют удаления. Медианная цена составляет 199 000, что является более надёжной характеристикой центра распределения, чем среднее значение (354 642), искажённое выбросами

In [3]:
df.head()

,id,type,sub_type,start_date,end_date,listing_type,tom,building_age,total_floor_count,floor_no,room_count,size,address,furnished,heating_type,price,price_currency
0,1,Konut,Rezidans,12/10/18,1/9/19,2,30,0,20 ve üzeri,2,2+1,90.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,3500.0,TRY
1,2,Konut,Daire,2/13/19,NaN,1,14,0,20 ve üzeri,20 ve üzeri,1+0,43.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,490000.0,TRY
2,3,Konut,Daire,10/9/18,11/8/18,1,30,0,1,Yüksek Giriş,2+1,NaN,Tekirdağ/Çorlu/Reşadiye,NaN,Fancoil,155000.0,TRY
3,4,Konut,Rezidans,9/10/18,10/10/18,1,30,3,20 ve üzeri,20 ve üzeri,6+1,450.0,İstanbul/Beşiktaş/Levent,NaN,Fancoil,32500000.0,TRY
4,5,Konut,Rezidans,12/10/18,1/9/19,1,30,0,20 ve üzeri,2,2+1,90.0,İstanbul/Kartal/Kordonboyu,NaN,Fancoil,1450000.0,TRY


In [80]:
df.isna().sum()

id                        0
type                      0
sub_type                  0
start_date                0
end_date             137189
listing_type              0
tom                       0
building_age          27390
total_floor_count     28021
floor_no              35296
room_count                0
size                 146006
address                   0
furnished            403487
heating_type          27970
price                   715
price_currency          715
dtype: int64

В датасете присутствуют пропуски в 9 из 17 столбцов. Наибольшее количество пропусков наблюдается в столбце furnished (403 487 — 100% данных), что делает этот столбец полностью пустым и непригодным для использования. 
Также значительное число пропусков зафиксировано в столбцах end_date (137 189), 
size (146 006) и floor_no (35 296).

Для дальнейшей удобной работы с дата сетом переведем его

# Перевод датасета

In [ ]:
import subprocess
import sys
import os
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "googletrans", "httpx", "-y"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "googletrans==4.0.0-rc1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "httpx==0.13.3"])
unique_texts = {}
for col in existing_columns:
    for val in df[col].dropna().unique():
        val_str = str(val).strip()
        if val_str and val_str not in unique_texts:
            unique_texts[val_str] = None
test_mode = False   
if test_mode:
    unique_items = list(unique_texts.items())[:100]
    print(f" ТЕСТОВЫЙ РЕЖИМ: {len(unique_items)} значений")
else:
    unique_items = list(unique_texts.items())
    
error_count = 0
success_count = 0
start_time = time.time()

for idx, (text, _) in enumerate(tqdm(unique_items, desc="Перевод")):
    try:
        
        result = translator.translate(text, src='tr', dest='ru')
        unique_texts[text] = result.text
        success_count += 1
        
        if idx % 5 == 0:
            time.sleep(0.1)
            
    except Exception as e:
        error_count += 1
        unique_texts[text] = '[ОШИБКА]'
        if error_count <= 3:
 
        elapsed = time.time() - start_time
        minutes = int(elapsed // 60)
        seconds = int(elapsed % 60)


for col in existing_columns:
    df[f'{col}_ru'] = df[col].astype(str).map(unique_texts).fillna('')

 
for idx in range(min(5, len(df))):
    print(f"\nСтрока {idx+1}:")
    for col in existing_columns[:3]:
        original = df.iloc[idx][col]
        translated = df.iloc[idx][f'{col}_ru']
        print(f"  {col}: {original} → {translated}")
output_file = 'translated_final.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n Сохранено: {output_file}")
print(f" {len(df):,} строк × {len(df.columns)} столбцов")

 

In [2]:
df2 = pd.read_csv('translated_final.csv')

In [3]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 403487 entries, 0 to 403486
Data columns (total 25 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 403487 non-null  int64  
 1   type               403487 non-null  object 
 2   sub_type           403487 non-null  object 
 3   start_date         403487 non-null  object 
 4   end_date           266298 non-null  object 
 5   listing_type       403487 non-null  int64  
 6   tom                403487 non-null  int64  
 7   building_age       376097 non-null  object 
 8   total_floor_count  375466 non-null  object 
 9   floor_no           368191 non-null  object 
 10  room_count         403487 non-null  object 
 11  size               257481 non-null  float64
 12  address            403487 non-null  object 
 13  furnished          0 non-null       float64
 14  heating_type       375517 non-null  object 
 15  price              402772 non-null  float64
 16  pr

In [ ]:
import pandas as pd
df = pd.read_csv('translated_final.csv', encoding='utf-8-sig')
df_clean = df[['id', 'type_ru', 'sub_type_ru', 'listing_type_ru', 'address_ru', 
               'heating_type_ru', 'building_age_ru', 'floor_no_ru', 'room_count', 
               'size', 'price', 'price_currency', 'tom', 'total_floor_count']].copy()

df_clean.columns = ['id', 'тип', 'подтип', 'тип_объявления', 'адрес', 
                    'отопление', 'возраст_здания', 'этаж', 'комнат', 
                    'площадь', 'цена', 'валюта', 'время_на_рынке', 'всего_этажей']
df_clean.to_csv('real_estate_clean.csv', index=False, encoding='utf-8-sig')
print(df_clean.head())

названия колонок переводим в ручную и оставляем только нужные нам признаки

In [9]:
df = pd.read_csv('real_estate_clean.csv')

In [6]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,комнат,площадь,цена,валюта,время_на_рынке,всего_этажей
0,1,Жилье,резиденция,2,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,3500.0,TRY,30,20 ve üzeri
1,2,Жилье,Квартира,1,Стамбул/Картал/Кордонбою,фанкойл,0,20 и выше,1+0,43.0,490000.0,TRY,14,20 ve üzeri
2,3,Жилье,Квартира,1,Текирдаг/Чорлу/Решадие,фанкойл,0,Высокий вход,2+1,NaN,155000.0,TRY,30,1
3,4,Жилье,резиденция,1,Стамбул/Бешикташ/Левент,фанкойл,3,20 и выше,6+1,450.0,32500000.0,TRY,30,20 ve üzeri
4,5,Жилье,резиденция,1,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,1450000.0,TRY,30,20 ve üzeri
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,2,Стамбул/Султанбейли/Адил,NaN,NaN,NaN,+,NaN,1500.0,TRY,162,NaN
403483,403484,Жилье,Квартира,1,Сакарья/Адапазары/Джумхуриет,NaN,NaN,NaN,2+1,NaN,120000.0,TRY,139,NaN
403484,403485,Жилье,Квартира,1,Анталья/Аланья/Сарай,NaN,NaN,NaN,1+1,NaN,48000.0,EUR,97,NaN
403485,403486,Жилье,Квартира,2,Айдын/Кушадасы/Туркмен,NaN,NaN,NaN,2+1,2.0,900.0,TRY,6,NaN


# Очистка и преобразование данных

In [11]:
mapping = {
    1: 'Продажа',
    2: 'Аренда'
}
df['тип_объявления'] = df['тип_объявления'].map(mapping)

переименовываем признаки в более удобный формат, у нас есть квартивы которые сдают в аренду и продают

In [14]:
df[['город', 'район', 'улица']] = df['адрес'].str.split('/', expand=True)


разделяем адресс на город район и улицу для более удобной работы

In [16]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,комнат,площадь,цена,валюта,время_на_рынке,всего_этажей,город,район,улица
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,3500.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20 и выше,1+0,43.0,490000.0,TRY,14,20 ve üzeri,Стамбул,Картал,Кордонбою
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,Высокий вход,2+1,NaN,155000.0,TRY,30,1,Текирдаг,Чорлу,Решадие
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20 и выше,6+1,450.0,32500000.0,TRY,30,20 ve üzeri,Стамбул,Бешикташ,Левент
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,1450000.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,NaN,NaN,NaN,+,NaN,1500.0,TRY,162,NaN,Стамбул,Султанбейли,Адил
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,NaN,NaN,NaN,2+1,NaN,120000.0,TRY,139,NaN,Сакарья,Адапазары,Джумхуриет
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,NaN,NaN,NaN,1+1,NaN,48000.0,EUR,97,NaN,Анталья,Аланья,Сарай
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,NaN,NaN,NaN,2+1,2.0,900.0,TRY,6,NaN,Айдын,Кушадасы,Туркмен


In [17]:
df['возраст_здания'].value_counts()

возраст_здания
0               140174
Между 6 и 10     50495
Между 11-15      32309
Между 16-20      31333
1                20355
4                19032
Между 21-25      18438
2                17466
3                15651
5                13589
Между 26-30      10581
Между 31-35       4268
Между 36-40       1347
40 и выше         1059
Name: count, dtype: int64

In [23]:
replacements = {
    'Между 6 и 10': 8,
    'Между 11-15': 13,
    'Между 16-20': 18,
    'Между 21-25': 23,
    'Между 26-30': 28,
    'Между 31-35': 33,
    'Между 36-40': 38,
    '40 и выше': 40
}

df['возраст_здания'] = df['возраст_здания'].replace(replacements)
df['возраст_здания'] = pd.to_numeric(df['возраст_здания'], errors='coerce').astype('Int64')

конвертируем возраст здания в более понятные значение 

In [27]:
df['возраст_здания'].isnull().sum()

np.int64(27390)

In [28]:
median_age = df['возраст_здания'].median()
df['возраст_здания'] = df['возраст_здания'].fillna(median_age)

в данном столбце есть 27390 пропусков заполним пропуски медианой 

In [45]:
for district in df['район'].unique():
    most_common = df[(df['район'] == district) & (df['отопление'] != 'Неизвестно')]['отопление'].mode()
    if not most_common.empty:
         mask = (df['район'] == district) & (df['отопление'] == 'Неизвестно')
        df.loc[mask, 'отопление'] = most_common[0]

 

в столбце отопление есть 27980 пропусков заполним их самым популярным видом отоплениям в их районе

In [50]:
df['этаж'].value_counts()

этаж
2                  65864
1                  46756
3                  30121
Высокий вход       24045
3                  22569
Частный            21165
4                  21049
садовый пол        19065
Первый этаж        13872
4                  13416
5                  12495
5                   8698
6                   5116
Джинсы 1            5036
джинсы 2            4987
9                   4855
6                   4631
8                   4608
7                   4398
10                  3863
джинсы 3            3793
Пентхаус            3566
7                   3300
Полный              2958
11                  2894
12                  2308
Джинсы 4            2269
13                  1702
20 и выше           1563
8                   1491
14                  1328
15                   911
Верхний этаж         894
Подвальный этаж      815
16                   600
17                   373
18                   334
Терраса Этаж         293
19                   177
Мезонин             

In [51]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,комнат,площадь,цена,валюта,время_на_рынке,всего_этажей,город,район,улица
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,3500.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20 и выше,1+0,43.0,490000.0,TRY,14,20 ve üzeri,Стамбул,Картал,Кордонбою
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,Высокий вход,2+1,NaN,155000.0,TRY,30,1,Текирдаг,Чорлу,Решадие
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20 и выше,6+1,450.0,32500000.0,TRY,30,20 ve üzeri,Стамбул,Бешикташ,Левент
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,1450000.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,NaN,+,NaN,1500.0,TRY,162,NaN,Стамбул,Султанбейли,Адил
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,NaN,2+1,NaN,120000.0,TRY,139,NaN,Сакарья,Адапазары,Джумхуриет
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,NaN,1+1,NaN,48000.0,EUR,97,NaN,Анталья,Аланья,Сарай
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,NaN,2+1,2.0,900.0,TRY,6,NaN,Айдын,Кушадасы,Туркмен


In [3]:
import pandas as pd

df = pd.read_csv('my_dataframe.csv', encoding='utf-8', low_memory=False)

df['этаж'] = df['этаж'].replace({
    'Джинсы 1': '0',
    'джинсы 2': '0',
    'джинсы 3': '0',
    'Джинсы 4': '0',
    'Высокий вход': '1',
    'Первый этаж': '1',
    'садовый пол': '-1',
    'Частный': '1',
    'Полный': '1',
    'Пентхаус': '-1',
    'Терраса Этаж': '-1',
    'Верхний этаж': 'последний',   
    'Подвальный этаж': '-2',
    'Мезонин': '0',
    '20 и выше': '20'
})

mask = df['этаж'] == 'последний'
df['этаж'] = df['этаж'].replace('последний', '5')

df['этаж'] = pd.to_numeric(df['этаж'], errors='coerce')

df['этаж'] = df['этаж'].fillna(df['этаж'].median())

df['этаж'] = df['этаж'].astype('Int64')

print("Результат:")
print(df['этаж'].value_counts().sort_index().head(15))



Результат:
этаж
-2       815
-1     22924
0      16097
1     108796
2     101160
3      52690
4      34465
5      22087
6       9747
7       7698
8       6099
9       4855
10      3863
11      2894
12      2309
Name: count, dtype: Int64


In [4]:
df['этаж'].isna().sum()

np.int64(0)

In [ ]:
переводим этажи в единый формат для последующей удобной работы

In [5]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,комнат,площадь,цена,валюта,время_на_рынке,всего_этажей,город,район,улица
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,3500.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,1+0,43.0,490000.0,TRY,14,20 ve üzeri,Стамбул,Картал,Кордонбою
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,2+1,NaN,155000.0,TRY,30,1,Текирдаг,Чорлу,Решадие
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,6+1,450.0,32500000.0,TRY,30,20 ve üzeri,Стамбул,Бешикташ,Левент
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,2+1,90.0,1450000.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,+,NaN,1500.0,TRY,162,NaN,Стамбул,Султанбейли,Адил
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,2+1,NaN,120000.0,TRY,139,NaN,Сакарья,Адапазары,Джумхуриет
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,1+1,NaN,48000.0,EUR,97,NaN,Анталья,Аланья,Сарай
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2+1,2.0,900.0,TRY,6,NaN,Айдын,Кушадасы,Туркмен


In [7]:
df['комнат'].value_counts()

комнат
3+1     157363
2+1     138677
1+1      39134
4+1      37472
5+1       8219
4+2       4540
5+2       2926
+         2898
3+2       2673
1+0       2612
6+1       2001
6+2       1517
2+2        836
7+1        523
7+2        428
10+0       313
8+1        218
8+2        189
6+3        167
5+3        139
4+3        133
7+3         86
9+1         71
9+3         65
9+2         61
8+3         55
8+4         38
10+1        35
10+2        30
9+5         27
9+4         26
10+3         9
10+5         2
0+0          1
10+4         1
11+3         1
15+5         1
Name: count, dtype: int64

In [42]:
def parse_rooms(value):
    if pd.isna(value) or value == '':
        return {'total': None, 'bedrooms': None, 'living': None}
    try:
        parts = str(value).split('+')
        if len(parts) == 2:
            b, l = int(parts[0]), int(parts[1])
            return {'total': b + l, 'bedrooms': b, 'living': l}
    except:
        pass
    return {'total': None, 'bedrooms': None, 'living': None}

rooms_data = df['комнат'].apply(parse_rooms).apply(pd.Series)

rooms_data['total'] = pd.to_numeric(rooms_data['total'], errors='coerce').astype('Int64')
rooms_data['bedrooms'] = pd.to_numeric(rooms_data['bedrooms'], errors='coerce').astype('Int64')
rooms_data['living'] = pd.to_numeric(rooms_data['living'], errors='coerce').astype('Int64')

df = pd.concat([df, rooms_data], axis=1)

df = df.rename(columns={
    'total': 'комнат_всего',
    'bedrooms': 'спальни',
    'living': 'гостиные'
})

print(df[['комнат', 'спальни', 'гостиные', 'комнат_всего']].head(10))
print(df[['спальни', 'гостиные', 'комнат_всего']].dtypes)

  комнат  спальни  гостиные  комнат_всего
0    2+1        2         1             3
1    1+0        1         0             1
2    2+1        2         1             3
3    6+1        6         1             7
4    2+1        2         1             3
5    1+1        1         1             2
6    3+1        3         1             4
7    4+1        4         1             5
8    3+1        3         1             4
9    2+2        2         2             4
спальни         Int64
гостиные        Int64
комнат_всего    Int64
dtype: object


Преобразуем турецский формат в более понятный Российский, сделаем отдельные колонки всего комнат спальни и гостинные,
это необходимо для дальнейшего обучения моделей

In [43]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,цена,валюта,время_на_рынке,всего_этажей,город,район,улица,комнат,комнат_всего,спальни,гостиные
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,3500.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою,2+1,3,2,1
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,490000.0,TRY,14,20 ve üzeri,Стамбул,Картал,Кордонбою,1+0,1,1,0
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,NaN,155000.0,TRY,30,1,Текирдаг,Чорлу,Решадие,2+1,3,2,1
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,32500000.0,TRY,30,20 ve üzeri,Стамбул,Бешикташ,Левент,6+1,7,6,1
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,1450000.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою,2+1,3,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,NaN,1500.0,TRY,162,NaN,Стамбул,Султанбейли,Адил,+,<NA>,<NA>,<NA>
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,NaN,120000.0,TRY,139,NaN,Сакарья,Адапазары,Джумхуриет,2+1,3,2,1
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,NaN,48000.0,EUR,97,NaN,Анталья,Аланья,Сарай,1+1,2,1,1
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,900.0,TRY,6,NaN,Айдын,Кушадасы,Туркмен,2+1,3,2,1


In [34]:
df


,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,цена,валюта,время_на_рынке,всего_этажей,город,район,улица
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,3500.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,490000.0,TRY,14,20 ve üzeri,Стамбул,Картал,Кордонбою
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,NaN,155000.0,TRY,30,1,Текирдаг,Чорлу,Решадие
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,32500000.0,TRY,30,20 ve üzeri,Стамбул,Бешикташ,Левент
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,1450000.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,NaN,1500.0,TRY,162,NaN,Стамбул,Султанбейли,Адил
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,NaN,120000.0,TRY,139,NaN,Сакарья,Адапазары,Джумхуриет
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,NaN,48000.0,EUR,97,NaN,Анталья,Аланья,Сарай
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,900.0,TRY,6,NaN,Айдын,Кушадасы,Туркмен


In [44]:
df['площадь'].isna().sum()

np.int64(146006)

In [46]:
import pandas as pd
import numpy as np

print(f"   Пустых в площади: {df['площадь'].isna().sum():,}")

df['группа'] = df['тип'] + '_' + df['комнат'].astype(str)

group_median = df.groupby('группа')['площадь'].median()

for group, median_val in group_median.items():
    mask = (df['группа'] == group) & (df['площадь'].isna())
    df.loc[mask, 'площадь'] = median_val

overall_median = df['площадь'].median()
df['площадь'] = df['площадь'].fillna(overall_median)

df = df.drop(columns=['группа'])

print(f"   Пустых в площади: {df['площадь'].isna().sum():,}")
print(f"   Всего строк: {len(df):,}")

   Пустых в площади: 0
   Пустых в площади: 0
   Всего строк: 403,487


для того чтобы заполнить пропущенные площади группируем по типу недвижимости и комнатам и в зависисмоти от этого заполняем пропуски

In [40]:
df['комнат'] = df1['комнат']


In [49]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,цена,валюта,время_на_рынке,всего_этажей,город,район,улица,комнат,комнат_всего,спальни,гостиные
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,3500.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою,2+1,3,2,1
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,490000.0,TRY,14,20 ve üzeri,Стамбул,Картал,Кордонбою,1+0,1,1,0
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,90.0,155000.0,TRY,30,1,Текирдаг,Чорлу,Решадие,2+1,3,2,1
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,32500000.0,TRY,30,20 ve üzeri,Стамбул,Бешикташ,Левент,6+1,7,6,1
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,1450000.0,TRY,30,20 ve üzeri,Стамбул,Картал,Кордонбою,2+1,3,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,280.0,1500.0,TRY,162,NaN,Стамбул,Султанбейли,Адил,+,<NA>,<NA>,<NA>
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,90.0,120000.0,TRY,139,NaN,Сакарья,Адапазары,Джумхуриет,2+1,3,2,1
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,56.0,48000.0,EUR,97,NaN,Анталья,Аланья,Сарай,1+1,2,1,1
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,900.0,TRY,6,NaN,Айдын,Кушадасы,Туркмен,2+1,3,2,1


In [52]:
# Конвертация цен в USD (без отдельного столбца с курсом)
df['цена_usd'] = df['цена'] * df['валюта'].map({'TRY': 1/34, 'USD': 1.0, 'EUR': 1.10, 'GBP': 1.30})

Конвертируем цену в единую валюту 

In [53]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,цена,...,время_на_рынке,всего_этажей,город,район,улица,комнат,комнат_всего,спальни,гостиные,цена_usd
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,3500.0,...,30,20 ve üzeri,Стамбул,Картал,Кордонбою,2+1,3,2,1,102.941176
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,490000.0,...,14,20 ve üzeri,Стамбул,Картал,Кордонбою,1+0,1,1,0,14411.764706
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,90.0,155000.0,...,30,1,Текирдаг,Чорлу,Решадие,2+1,3,2,1,4558.823529
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,32500000.0,...,30,20 ve üzeri,Стамбул,Бешикташ,Левент,6+1,7,6,1,955882.352941
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,1450000.0,...,30,20 ve üzeri,Стамбул,Картал,Кордонбою,2+1,3,2,1,42647.058824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,280.0,1500.0,...,162,NaN,Стамбул,Султанбейли,Адил,+,<NA>,<NA>,<NA>,44.117647
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,90.0,120000.0,...,139,NaN,Сакарья,Адапазары,Джумхуриет,2+1,3,2,1,3529.411765
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,56.0,48000.0,...,97,NaN,Анталья,Аланья,Сарай,1+1,2,1,1,52800.000000
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,900.0,...,6,NaN,Айдын,Кушадасы,Туркмен,2+1,3,2,1,26.470588


In [54]:
df['всего_этажей'].value_counts()

всего_этажей
4              83082
3              77956
5              70104
10-20 arası    36512
2              27742
6              23348
10             12558
7              12284
8              11207
9               9029
20 ve üzeri     6679
1               4965
Name: count, dtype: int64

In [55]:
 
floor_map = {
    '20 ve üzeri': 25,
    '10-20 arası': 15,
     
}
df['всего_этажей'] = df['всего_этажей'].replace(floor_map)
df['всего_этажей'] = pd.to_numeric(df['всего_этажей'], errors='coerce')

 

In [56]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,цена,...,время_на_рынке,всего_этажей,город,район,улица,комнат,комнат_всего,спальни,гостиные,цена_usd
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,3500.0,...,30,25.0,Стамбул,Картал,Кордонбою,2+1,3,2,1,102.941176
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,490000.0,...,14,25.0,Стамбул,Картал,Кордонбою,1+0,1,1,0,14411.764706
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,90.0,155000.0,...,30,1.0,Текирдаг,Чорлу,Решадие,2+1,3,2,1,4558.823529
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,32500000.0,...,30,25.0,Стамбул,Бешикташ,Левент,6+1,7,6,1,955882.352941
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,1450000.0,...,30,25.0,Стамбул,Картал,Кордонбою,2+1,3,2,1,42647.058824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,280.0,1500.0,...,162,NaN,Стамбул,Султанбейли,Адил,+,<NA>,<NA>,<NA>,44.117647
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,90.0,120000.0,...,139,NaN,Сакарья,Адапазары,Джумхуриет,2+1,3,2,1,3529.411765
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,56.0,48000.0,...,97,NaN,Анталья,Аланья,Сарай,1+1,2,1,1,52800.000000
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,900.0,...,6,NaN,Айдын,Кушадасы,Туркмен,2+1,3,2,1,26.470588


In [57]:
df['всего_этажей'] = df['всего_этажей'].astype('Int64')

In [59]:
df['цена_usd'] = df['цена_usd'].round(2)


In [60]:
df

,id,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,цена,...,время_на_рынке,всего_этажей,город,район,улица,комнат,комнат_всего,спальни,гостиные,цена_usd
0,1,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,3500.0,...,30,25,Стамбул,Картал,Кордонбою,2+1,3,2,1,102.94
1,2,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,490000.0,...,14,25,Стамбул,Картал,Кордонбою,1+0,1,1,0,14411.76
2,3,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,90.0,155000.0,...,30,1,Текирдаг,Чорлу,Решадие,2+1,3,2,1,4558.82
3,4,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,32500000.0,...,30,25,Стамбул,Бешикташ,Левент,6+1,7,6,1,955882.35
4,5,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,1450000.0,...,30,25,Стамбул,Картал,Кордонбою,2+1,3,2,1,42647.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,403483,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,280.0,1500.0,...,162,<NA>,Стамбул,Султанбейли,Адил,+,<NA>,<NA>,<NA>,44.12
403483,403484,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,90.0,120000.0,...,139,<NA>,Сакарья,Адапазары,Джумхуриет,2+1,3,2,1,3529.41
403484,403485,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,56.0,48000.0,...,97,<NA>,Анталья,Аланья,Сарай,1+1,2,1,1,52800.00
403485,403486,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,900.0,...,6,<NA>,Айдын,Кушадасы,Туркмен,2+1,3,2,1,26.47


In [61]:
df = df.drop(columns=['id'])

In [62]:
df = df.drop(columns=['цена'])

In [63]:
df = df.drop(columns=['комнат'])

In [64]:
df

,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,валюта,время_на_рынке,всего_этажей,город,район,улица,комнат_всего,спальни,гостиные,цена_usd
0,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,TRY,30,25,Стамбул,Картал,Кордонбою,3,2,1,102.94
1,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,TRY,14,25,Стамбул,Картал,Кордонбою,1,1,0,14411.76
2,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,90.0,TRY,30,1,Текирдаг,Чорлу,Решадие,3,2,1,4558.82
3,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,TRY,30,25,Стамбул,Бешикташ,Левент,7,6,1,955882.35
4,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,TRY,30,25,Стамбул,Картал,Кордонбою,3,2,1,42647.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403482,Жилье,Квартира,Аренда,Стамбул/Султанбейли/Адил,Комбинированный котел (природный газ),3,2,280.0,TRY,162,<NA>,Стамбул,Султанбейли,Адил,<NA>,<NA>,<NA>,44.12
403483,Жилье,Квартира,Продажа,Сакарья/Адапазары/Джумхуриет,Комбинированный котел (природный газ),3,2,90.0,TRY,139,<NA>,Сакарья,Адапазары,Джумхуриет,3,2,1,3529.41
403484,Жилье,Квартира,Продажа,Анталья/Аланья/Сарай,Кондиционер,3,2,56.0,EUR,97,<NA>,Анталья,Аланья,Сарай,2,1,1,52800.00
403485,Жилье,Квартира,Аренда,Айдын/Кушадасы/Туркмен,Кондиционер,3,2,2.0,TRY,6,<NA>,Айдын,Кушадасы,Туркмен,3,2,1,26.47


In [67]:
print("ПРОПУСКИ ПО СТОЛБЦАМ:")
print(df.isna().sum().sort_values(ascending=False))

print("ПРОЦЕНТ ПРОПУСКОВ:")
print((df.isna().sum() / len(df) * 100).sort_values(ascending=False).round(2))

ПРОПУСКИ ПО СТОЛБЦАМ:
всего_этажей      28021
комнат_всего       2898
гостиные           2898
спальни            2898
тип_объявления     2242
улица              1458
район              1449
валюта              715
цена_usd            715
тип                   0
подтип                0
площадь               0
время_на_рынке        0
возраст_здания        0
адрес                 0
отопление             0
этаж                  0
город                 0
dtype: int64
ПРОЦЕНТ ПРОПУСКОВ:
всего_этажей      6.94
комнат_всего      0.72
гостиные          0.72
спальни           0.72
тип_объявления    0.56
улица             0.36
район             0.36
валюта            0.18
цена_usd          0.18
тип               0.00
подтип            0.00
площадь           0.00
время_на_рынке    0.00
возраст_здания    0.00
адрес             0.00
отопление         0.00
этаж              0.00
город             0.00
dtype: float64


после обработки данных остались еще пропуски в данных их меньше 6% можем их удалить

In [70]:
df = df.dropna()

удаляем оставшиеся пропуски

In [73]:
df.to_csv('result_file.csv')

In [85]:
df = pd.read_csv('result_file.csv')

In [86]:
df

,Unnamed: 0,тип,подтип,тип_объявления,адрес,отопление,возраст_здания,этаж,площадь,валюта,время_на_рынке,всего_этажей,город,район,улица,комнат_всего,спальни,гостиные,цена_usd
0,0,Жилье,резиденция,Аренда,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,TRY,30,25,Стамбул,Картал,Кордонбою,3,2,1,102.94
1,1,Жилье,Квартира,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,20,43.0,TRY,14,25,Стамбул,Картал,Кордонбою,1,1,0,14411.76
2,2,Жилье,Квартира,Продажа,Текирдаг/Чорлу/Решадие,фанкойл,0,1,90.0,TRY,30,1,Текирдаг,Чорлу,Решадие,3,2,1,4558.82
3,3,Жилье,резиденция,Продажа,Стамбул/Бешикташ/Левент,фанкойл,3,20,450.0,TRY,30,25,Стамбул,Бешикташ,Левент,7,6,1,955882.35
4,4,Жилье,резиденция,Продажа,Стамбул/Картал/Кордонбою,фанкойл,0,2,90.0,TRY,30,25,Стамбул,Картал,Кордонбою,3,2,1,42647.06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369741,394882,Жилье,Семейный дом,Продажа,ТРСК/Искеле/Богаз Мх.,Кондиционер,28,2,90.0,GBP,32,1,ТРСК,Искеле,Богаз Мх.,3,2,1,45500.00
369742,394902,Жилье,Сборный дом,Продажа,Стамбул/Пендик/Кайнарджа,Комбинированный котел (природный газ),3,2,40.0,TRY,30,1,Стамбул,Пендик,Кайнарджа,2,1,1,926.47
369743,394916,Жилье,Семейный дом,Продажа,Мугла/Бодрум/Гюндоган,Кондиционер,0,2,315.0,TRY,60,2,Мугла,Бодрум,Гюндоган,7,5,2,161764.71
369744,394923,Жилье,Квартира,Аренда,Мугла/Мармарис/Кемералты,Кондиционер,18,2,56.0,TRY,2,4,Мугла,Мармарис,Кемералты,2,1,1,23.53
